# Analytics Layer - DuckDB SQL Query Execution

This notebook loads the Gold layer star schema tables into DuckDB and executes analytical SQL queries to generate business insights.

## 1. Import Required Libraries

In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import json
from datetime import datetime

# Setup paths
work_dir = Path('/home/jovyan/work')
gold_dir = work_dir / 'output' / 'gold'
submission_dir = Path('/home/jovyan/work')

print(f"✓ Gold layer directory: {gold_dir}")
print(f"✓ Files available: {list(gold_dir.glob('*.csv'))}")

✓ Gold layer directory: /home/jovyan/work/output/gold
✓ Files available: [PosixPath('/home/jovyan/work/output/gold/dim_customers.csv'), PosixPath('/home/jovyan/work/output/gold/fact_order_items.csv'), PosixPath('/home/jovyan/work/output/gold/dim_sellers.csv'), PosixPath('/home/jovyan/work/output/gold/dim_products.csv')]


## 2. Connect to DuckDB and Load Gold Layer Tables

In [2]:
# Connect to DuckDB (in-memory database)
con = duckdb.connect(':memory:')
print("✓ Connected to DuckDB (in-memory)")

# Load gold layer CSVs into DuckDB tables
tables = ['fact_order_items', 'dim_customers', 'dim_products', 'dim_sellers']
table_stats = {}

for table_name in tables:
    csv_path = gold_dir / f'{table_name}.csv'
    if csv_path.exists():
        # Use DuckDB's built-in CSV reading
        con.execute(f"CREATE TABLE {table_name} AS SELECT * FROM read_csv_auto('{csv_path}')")
        
        # Get table stats
        result = con.execute(f"SELECT COUNT(*) as row_count FROM {table_name}").fetchall()
        row_count = result[0][0]
        table_stats[table_name] = row_count
        print(f"✓ Loaded {table_name}: {row_count} rows")
    else:
        print(f"✗ Missing {csv_path}")

print(f"\n{json.dumps(table_stats, indent=2)}")

✓ Connected to DuckDB (in-memory)
✓ Loaded fact_order_items: 1000 rows
✓ Loaded dim_customers: 1000 rows
✓ Loaded dim_products: 1000 rows
✓ Loaded dim_sellers: 1000 rows

{
  "fact_order_items": 1000,
  "dim_customers": 1000,
  "dim_products": 1000,
  "dim_sellers": 1000
}


## 3. Query 1: Revenue Trend Analysis

In [3]:
# Read the SQL query from the submission directory
query_file = submission_dir / 'sql' / 'query_1_revenue_trend_analysis.sql'

if query_file.exists():
    with open(query_file, 'r') as f:
        query = f.read()
    print(f"✓ Loaded query from {query_file}\n")
    print("Query Preview:")
    print("=" * 80)
    print(query[:500] + "..." if len(query) > 500 else query)
    print("=" * 80)
else:
    print(f"✗ Query file not found: {query_file}")

✓ Loaded query from /home/jovyan/work/sql/query_1_revenue_trend_analysis.sql

Query Preview:
-- Query 1: Revenue Trend Analysis
-- Analyzes revenue trends over time by month, including total revenue, average order value, and delivery performance

SELECT
    DATE_TRUNC('month', f.order_date) as order_month,
    COUNT(DISTINCT f.order_id) as total_orders,
    COUNT(f.order_item_sk) as total_items_sold,
    SUM(f.price + f.freight_value) as gross_revenue,
    SUM(f.payment_value) as total_payment_received,
    AVG(f.price + f.freight_value) as avg_item_value,
    AVG(f.days_to_deliver) as ...


In [4]:
# Execute the query
try:
    result = con.execute(query).fetchall()
    columns = [desc[0] for desc in con.description]
    
    # Convert to pandas DataFrame for better display
    df_results = pd.DataFrame(result, columns=columns)
    
    print(f"✓ Query executed successfully")
    print(f"✓ Result: {len(df_results)} rows, {len(df_results.columns)} columns\n")
    
    # Display full results
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    print(df_results.to_string())
    
except Exception as e:
    print(f"✗ Query execution failed: {str(e)}")

✓ Query executed successfully
✓ Result: 2 rows, 11 columns

  order_month  total_orders  total_items_sold  gross_revenue  total_payment_received  avg_item_value  avg_delivery_days  late_delivery_rate_pct  unique_customers  unique_products_sold  unique_sellers
0  2018-07-01            27                28        3516.33                     NaN      125.583214           7.142857                    0.00                 0                     2              12
1  2018-08-01            29                31        4331.24                  138.84      139.717419           8.366667                    3.23                 0                     2              12


## 4. Export Results to CSV

In [5]:
# Create output directory for query results
output_dir = submission_dir / 'output'
output_dir.mkdir(exist_ok=True)

# Export results to CSV with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f'query_1_revenue_trend_analysis_{timestamp}.csv'

df_results.to_csv(output_file, index=False)
print(f"✓ Results exported to: {output_file}")
print(f"✓ File size: {output_file.stat().st_size / 1024:.2f} KB")

# Also save a JSON summary
summary = {
    'query': 'Revenue Trend Analysis',
    'executed_at': datetime.now().isoformat(),
    'total_rows': len(df_results),
    'columns': list(df_results.columns),
    'output_file': str(output_file)
}

summary_file = output_dir / f'query_1_summary_{timestamp}.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"✓ Summary saved to: {summary_file}")

✓ Results exported to: /home/jovyan/work/output/query_1_revenue_trend_analysis_20260323_144727.csv
✓ File size: 0.36 KB
✓ Summary saved to: /home/jovyan/work/output/query_1_summary_20260323_144727.json


## 5. Schema Verification - Gold Layer Tables

In [6]:
# Display schema for all tables
print("=" * 80)
print("GOLD LAYER TABLE SCHEMAS")
print("=" * 80)

for table_name in tables:
    try:
        schema_result = con.execute(f"DESCRIBE {table_name}").fetchall()
        print(f"\n{table_name}:")
        print("-" * 40)
        for col_info in schema_result:
            col_name, col_type = col_info[0], col_info[1]
            print(f"  {col_name:<30} {col_type}")
    except Exception as e:
        print(f"✗ Error describing {table_name}: {str(e)}")

print("\n" + "=" * 80)

GOLD LAYER TABLE SCHEMAS

fact_order_items:
----------------------------------------
  order_item_sk                  BIGINT
  order_id                       VARCHAR
  order_item_id                  BIGINT
  customer_key                   VARCHAR
  product_key                    BIGINT
  seller_key                     BIGINT
  order_date                     DATE
  order_status                   VARCHAR
  price                          DOUBLE
  freight_value                  DOUBLE
  payment_value                  DOUBLE
  payment_type                   VARCHAR
  payment_installments_max       BIGINT
  days_to_deliver                BIGINT
  days_delivery_vs_estimate      BIGINT
  is_late_delivery               BOOLEAN

dim_customers:
----------------------------------------
  customer_key                   BIGINT
  customer_unique_id             VARCHAR
  customer_city                  VARCHAR
  customer_state                 VARCHAR
  customer_zip_code_prefix       BIGINT
  first_orde

## 6. Data Quality Summary

In [7]:
print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

for table_name in tables:
    try:
        # Count total rows
        count_result = con.execute(f"SELECT COUNT(*) as total FROM {table_name}").fetchall()
        total_rows = count_result[0][0]
        
        # Get column list
        schema_result = con.execute(f"DESCRIBE {table_name}").fetchall()
        columns = [col[0] for col in schema_result]
        
        print(f"\n{table_name}:")
        print(f"  Total rows: {total_rows}")
        print(f"  Columns: {len(columns)}")
        
        # Check for nulls
        null_counts = {}
        for col in columns:
            null_result = con.execute(f"SELECT COUNT(*) FROM {table_name} WHERE {col} IS NULL").fetchall()
            null_count = null_result[0][0]
            if null_count > 0:
                null_counts[col] = null_count
        
        if null_counts:
            print(f"  Null values detected:")
            for col, count in null_counts.items():
                pct = (count / total_rows * 100) if total_rows > 0 else 0
                print(f"    - {col}: {count} ({pct:.2f}%)")
        else:
            print(f"  ✓ No null values detected")
            
    except Exception as e:
        print(f"✗ Error analyzing {table_name}: {str(e)}")

print("\n" + "=" * 80)
print("✓ Analytics pipeline completed successfully!")
print("=" * 80)

DATA QUALITY SUMMARY

fact_order_items:
  Total rows: 1000
  Columns: 16
  Null values detected:
    - customer_key: 1000 (100.00%)
    - product_key: 974 (97.40%)
    - seller_key: 666 (66.60%)
    - order_date: 941 (94.10%)
    - order_status: 941 (94.10%)
    - payment_value: 992 (99.20%)
    - payment_type: 992 (99.20%)
    - payment_installments_max: 992 (99.20%)
    - days_to_deliver: 942 (94.20%)
    - days_delivery_vs_estimate: 942 (94.20%)
    - is_late_delivery: 942 (94.20%)

dim_customers:
  Total rows: 1000
  Columns: 9
  Null values detected:
    - first_order_date: 995 (99.50%)
    - total_orders: 995 (99.50%)
    - total_spend: 995 (99.50%)

dim_products:
  Total rows: 1000
  Columns: 7
  Null values detected:
    - product_photos_qty: 29 (2.90%)
    - product_description_lenght: 29 (2.90%)

dim_sellers:
  Total rows: 1000
  Columns: 5
  ✓ No null values detected

✓ Analytics pipeline completed successfully!


## 7. Query 2: Seller Performance Scorecard

In [8]:
# Read and execute query 2
query_2_file = submission_dir / 'sql' / 'query_2_seller_performance_scorecard.sql'

if query_2_file.exists():
    with open(query_2_file, 'r') as f:
        query_2 = f.read()
    print(f"✓ Loaded query from {query_2_file}\n")
    print("Query Preview:")
    print("=" * 80)
    print(query_2[:500] + "..." if len(query_2) > 500 else query_2)
    print("=" * 80)
else:
    print(f"✗ Query file not found: {query_2_file}")

✓ Loaded query from /home/jovyan/work/sql/query_2_seller_performance_scorecard.sql

Query Preview:
-- Query 2: Seller Performance Scorecard
-- Ranks sellers by performance metrics including revenue, order volume, customer satisfaction, and delivery performance

SELECT
    s.seller_key,
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT f.order_id) as total_orders,
    COUNT(f.order_item_sk) as total_items_sold,
    SUM(f.price + f.freight_value) as total_revenue,
    ROUND(AVG(f.price + f.freight_value), 2) as avg_item_value,
    COUNT(DISTINCT f.customer_key) as uniqu...


In [9]:
# Execute query 2
try:
    result_2 = con.execute(query_2).fetchall()
    columns_2 = [desc[0] for desc in con.description]
    
    # Convert to pandas DataFrame
    df_results_2 = pd.DataFrame(result_2, columns=columns_2)
    
    print(f"✓ Query executed successfully")
    print(f"✓ Result: {len(df_results_2)} rows, {len(df_results_2.columns)} columns\n")
    
    # Display results
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    print(df_results_2.to_string())
    
except Exception as e:
    print(f"✗ Query execution failed: {str(e)}")

✓ Query executed successfully
✓ Result: 22 rows, 14 columns

    seller_key                         seller_id            seller_city seller_state  total_orders  total_items_sold  total_revenue  avg_item_value  unique_customers  avg_items_per_order  avg_delivery_days  late_delivery_rate_pct  successful_delivery_rate_pct  revenue_rank
0          594  9c4d31c7e46ab03a43fc06e3142afd4e         rio de janeiro           RJ             1                 1         792.17          792.17                 0                  0.0                7.0                     0.0                         100.0             1
1          311  53243585a1d6dc2643021fd1853d8905       lauro de freitas           BA             2                 2         503.66          251.83                 0                  0.0                8.5                     0.0                         100.0             2
2          334  59b22a78efb79a4797979612b885db36             uberlandia           MG             1                 1 

In [10]:
# Export query 2 results to CSV
timestamp_2 = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file_2 = output_dir / f'query_2_seller_performance_scorecard_{timestamp_2}.csv'

df_results_2.to_csv(output_file_2, index=False)
print(f"✓ Results exported to: {output_file_2}")
print(f"✓ File size: {output_file_2.stat().st_size / 1024:.2f} KB")

# Save JSON summary
summary_2 = {
    'query': 'Seller Performance Scorecard',
    'executed_at': datetime.now().isoformat(),
    'total_rows': len(df_results_2),
    'columns': list(df_results_2.columns),
    'output_file': str(output_file_2)
}

summary_file_2 = output_dir / f'query_2_summary_{timestamp_2}.json'
with open(summary_file_2, 'w') as f:
    json.dump(summary_2, f, indent=2, default=str)

print(f"✓ Summary saved to: {summary_file_2}")

✓ Results exported to: /home/jovyan/work/output/query_2_seller_performance_scorecard_20260323_145148.csv
✓ File size: 2.27 KB
✓ Summary saved to: /home/jovyan/work/output/query_2_summary_20260323_145148.json
